In [1]:
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns

train=pd.read_csv('/kaggle/input/competitions/titanic/train.csv', header=0)
test=pd.read_csv('/kaggle/input/competitions/titanic/test.csv', header=0)
gender_submission=pd.read_csv('/kaggle/input/competitions/titanic/gender_submission.csv', header=0)
target=train['Survived']

In [2]:
def transform(train):
    train['Age']=train['Age'].fillna(train['Age'].median())
    train['Age']=train['Age'].astype(float)

    train['Embarked']=train['Embarked'].fillna('Q')

    train['FareNormalized']=np.log1p(train['Fare'])

    train['FamilySize']=train['SibSp']+train['Parch']+1
    train['IsAlone']=(train['FamilySize']==1)

    train['IsAlone']=train['IsAlone'].astype(int)

    sexDict={'male': 0, 'female': 1}
    train['Sex']=train['Sex'].map(sexDict)

    embDict={'S': 0, 'C': 1, 'Q': 2}
    train['Embarked']=train['Embarked'].map(embDict)

    colList=['Pclass', 'Sex', 'Age', 'FareNormalized', 'FamilySize', 'IsAlone', 'Embarked']
    train=train[colList]

    return train

In [3]:
df=transform(train)
df_test=transform(test)


In [4]:
import lightgbm as lgb
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test=train_test_split(df, target, test_size=0.2, stratify=target)

In [5]:
def modelSelection(X_train, Y_train, X_test, Y_test):

    model=lgb.LGBMClassifier(
        boosting_type='gbdt', n_estimators=15, \
        learning_rate=0.05, num_leaves=31, verbose=-1
    )
    
    model.fit(X_train, Y_train)
    Y_pred=model.predict(X_test)
    acc=accuracy_score(Y_pred, Y_test)
    return model, acc

In [6]:
model, acc=modelSelection(X_train, Y_train, X_test, Y_test)
predictions=model.predict(df_test)

In [7]:
ans=pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': predictions
})

In [8]:
ans.to_csv('/kaggle/working/Submission_ship.csv', index=False)